In [ ]:
# Phase 1 — Station Audit and Data Quality Scoring

## Step 1.1 — Namibian Station Identification

Correct GHCND station IDs were obtained directly from the NOAA master stations list at:
https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt

| Station Name        | Original ISD ID  | Confirmed GHCND ID  | Records  |
|---------------------|------------------|---------------------|----------|
| Windhoek (main/GSN) | 68110099999      | WA007401540         | 19,793   |
| Windhoek (alt A)    | 68110099999      | WA007400630         | 5,084    |
| Windhoek (alt B)    | 68110099999      | WA007401240         | 5,539    |
| Windhoek (alt C)    | 68110099999      | WA007401850         | 5,175    |
| Walvis Bay Airport  | 68066099999      | WAM00068098         | 7,429    |
| Gobabis             | 68108099999      | WA007878380         | 12,242   |
| Keetmanshoop        | 68104099999      | WA004192150         | 5,236    |
| Ondangwa            | 68006099999      | WAM00068006         | 6,642    |
| Rundu               | 68004099999      | WA012084750         | 17,121   |
| Mariental           | 68106099999      | WA005688170         | 5,769    |
| Lüderitz            | 68096099999      | **NOT IN GHCND**    | —        |

**Note:** Lüderitz is absent from the GHCND database entirely. 
This is a known data gap and will be flagged in the station audit memo.
The primary station for EVT analysis is: WA007401540 (Windhoek GSN, 1970–2024).

In [ ]:
import requests

# Downloading the official GHCND stations list and search for Namibia
print("Downloading GHCND stations list...")
r = requests.get('https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt')

lines = r.text.split('\n')

# Namibia stations start with 'SWA' (South West Africa — the historical name)
# or 'WA' in the FIPS system
namibia_lines = [l for l in lines if l.startswith('SWA') or 
                 'WINDHOEK' in l.upper() or 
                 'WALVIS' in l.upper() or
                 'NAMIBIA' in l.upper() or
                 'GOBABIS' in l.upper() or
                 'KEETMANSHOOP' in l.upper() or
                 'ONDANGWA' in l.upper() or
                 'LUDERITZ' in l.upper() or
                 'RUNDU' in l.upper() or
                 'MARIENTAL' in l.upper()]

print(f"\nFound {len(namibia_lines)} Namibia stations:\n")
for l in namibia_lines:
    print(l)


Found 13 Namibia stations:

ASN00062004 -32.6000  149.6000  498.3    BURRUNDULLA                                 
BR047348180 -12.0500  -42.8000  490.0    BRUNDUE                                     
IN020030800  11.2800   77.5800  294.0    PERUNDURAI                                  
WA004192150 -26.5800   18.1300  980.0    KEETMANSHOOP                                
WA005688170 -24.6200   17.9700 1110.0    MARIENTAL                                   
WA007400630 -22.5500   17.0500 1640.0    WINDHOEK                                    
WA007401240 -22.5700   17.0800 1660.0    WINDHOEK                                    
WA007401540 -22.5670   17.1000 1700.0    WINDHOEK                       GSN     68110
WA007401850 -22.5800   17.1200 1740.0    WINDHOEK                                    
WA007878380 -22.5000   18.9670 1400.0    GOBABIS                                68116
WA012084750 -17.9170   19.7670 1100.0    RUNDU                                  68018
WAM00068006 -17.9330   15

In [ ]:
import time

# This are the correct station IDs with NO 'GHCND:' prefix for this API endpoint
STATIONS = {
    'windhoek_main' : 'WA007401540',  # GSN station — best record
    'windhoek_a'    : 'WA007400630',
    'windhoek_b'    : 'WA007401240',
    'windhoek_c'    : 'WA007401850',
    'gobabis'       : 'WA007878380',
    'keetmanshoop'  : 'WA004192150',
    'mariental'     : 'WA005688170',
    'rundu'         : 'WA012084750',
    'ondangwa'      : 'WAM00068006',
    'walvis_bay'    : 'WAM00068098',
}

DATE_CHUNKS = [
    ('1970-01-01', '1979-12-31'),
    ('1980-01-01', '1989-12-31'),
    ('1990-01-01', '1999-12-31'),
    ('2000-01-01', '2009-12-31'),
    ('2010-01-01', '2019-12-31'),
    ('2020-01-01', '2024-12-31'),
]

for name, station_id in STATIONS.items():
    print(f"\nFetching {name} ({station_id})...")
    all_chunks = []

    for start, end in DATE_CHUNKS:
        params = {
            'dataset'   : 'daily-summaries',
            'stations'  : station_id,
            'startDate' : start,
            'endDate'   : end,
            'dataTypes' : 'PRCP,TMAX,TMIN,AWND',
            'format'    : 'json',
            'units'     : 'metric',
        }

        resp = requests.get(BASE, params=params, headers=headers)

        if resp.status_code == 200:
            try:
                data = resp.json()
                if len(data) > 0:
                    all_chunks.append(pd.DataFrame(data))
                    print(f"  ✓ {start[:4]}s: {len(data)} records")
                else:
                    print(f"  ⚠ {start[:4]}s: no data")
            except Exception as e:
                print(f"  ✗ {start[:4]}s: error — {e}")
        else:
            print(f"  ✗ {start[:4]}s: status {resp.status_code}")

        time.sleep(1)

    if all_chunks:
        df_full = pd.concat(all_chunks, ignore_index=True)
        df_full['DATE'] = pd.to_datetime(df_full['DATE'])
        df_full.to_csv(f'../data/raw/{name}_daily.csv', index=False)
        print(f"  → TOTAL: {len(df_full)} records saved to data/raw/{name}_daily.csv")
    else:
        print(f"  → No data retrieved for {name}")

print("\nAll stations done!")


Fetching windhoek_main (WA007401540)...
  ✓ 1970s: 3652 records
  ✓ 1980s: 3653 records
  ✓ 1990s: 3652 records
  ✓ 2000s: 3559 records
  ✓ 2010s: 3515 records
  ✓ 2020s: 1762 records
  → TOTAL: 19793 records saved to data/raw/windhoek_main_daily.csv

Fetching windhoek_a (WA007400630)...
  ✓ 1970s: 2831 records
  ✓ 1980s: 2253 records
  ⚠ 1990s: no data
  ⚠ 2000s: no data
  ⚠ 2010s: no data
  ⚠ 2020s: no data
  → TOTAL: 5084 records saved to data/raw/windhoek_a_daily.csv

Fetching windhoek_b (WA007401240)...
  ✓ 1970s: 3286 records
  ✓ 1980s: 2253 records
  ⚠ 1990s: no data
  ⚠ 2000s: no data
  ⚠ 2010s: no data
  ⚠ 2020s: no data
  → TOTAL: 5539 records saved to data/raw/windhoek_b_daily.csv

Fetching windhoek_c (WA007401850)...
  ✓ 1970s: 2922 records
  ✓ 1980s: 2253 records
  ⚠ 1990s: no data
  ⚠ 2000s: no data
  ⚠ 2010s: no data
  ⚠ 2020s: no data
  → TOTAL: 5175 records saved to data/raw/windhoek_c_daily.csv

Fetching gobabis (WA007878380)...
  ✓ 1970s: 3515 records
  ✓ 1980s: 220

In [12]:
import numpy as np

def data_quality_score(df, station_name):
    """
    Computes a scalar data quality score [0,1] for a station record.
    0 = no usable data. 1 = full, clean, long record.
    """
    df = df.copy()
    df = df.set_index('DATE').sort_index()

    # --- Component 1: Record length (weight 0.35)
    years = (df.index[-1] - df.index[0]).days / 365.25
    length_score = min(years / 30, 1.0)  # 30 yrs = full score

    # --- Component 2: Completeness (weight 0.40)
    expected_days = (df.index[-1] - df.index[0]).days + 1
    actual_days   = len(df)
    completeness  = actual_days / expected_days

    # --- Component 3: PRCP non-null rate (weight 0.25)
    if 'PRCP' in df.columns:
        prcp_rate = df['PRCP'].notna().mean()
    else:
        prcp_rate = 0.0

    # --- Weighted composite
    score = (0.35 * length_score) + (0.40 * completeness) + (0.25 * prcp_rate)

    print(f"{station_name}")
    print(f"  Record length : {years:.1f} years  -> score {length_score:.2f}")
    print(f"  Completeness  : {completeness:.3f}         -> score {completeness:.2f}")
    print(f"  PRCP coverage : {prcp_rate:.3f}         -> score {prcp_rate:.2f}")
    print(f"  COMPOSITE     : {score:.3f}")
    print()
    return round(score, 3)


# --- Run on all stations
STATION_FILES = {
    'Windhoek Main (GSN)' : '../data/raw/windhoek_main_daily.csv',
    'Windhoek A'          : '../data/raw/windhoek_a_daily.csv',
    'Windhoek B'          : '../data/raw/windhoek_b_daily.csv',
    'Windhoek C'          : '../data/raw/windhoek_c_daily.csv',
    'Gobabis'             : '../data/raw/gobabis_daily.csv',
    'Keetmanshoop'        : '../data/raw/keetmanshoop_daily.csv',
    'Mariental'           : '../data/raw/mariental_daily.csv',
    'Rundu'               : '../data/raw/rundu_daily.csv',
    'Ondangwa'            : '../data/raw/ondangwa_daily.csv',
    'Walvis Bay'          : '../data/raw/walvis_bay_daily.csv',
}

dqs_results = {}

for station_name, filepath in STATION_FILES.items():
    df = pd.read_csv(filepath, parse_dates=['DATE'])
    score = data_quality_score(df, station_name)
    dqs_results[station_name] = score

# --- Summary table
print("=" * 50)
print("DATA QUALITY SCORE SUMMARY")
print("=" * 50)
for name, score in dqs_results.items():
    flag = "✅" if score >= 0.50 else "⚠ BELOW THRESHOLD"
    print(f"  {name:<25} DQS = {score:.3f}  {flag}")

Windhoek Main (GSN)
  Record length : 55.0 years  -> score 1.00
  Completeness  : 0.985         -> score 0.99
  PRCP coverage : 0.706         -> score 0.71
  COMPOSITE     : 0.921

Windhoek A
  Record length : 14.4 years  -> score 0.48
  Completeness  : 0.966         -> score 0.97
  PRCP coverage : 1.000         -> score 1.00
  COMPOSITE     : 0.804

Windhoek B
  Record length : 15.4 years  -> score 0.51
  Completeness  : 0.984         -> score 0.98
  PRCP coverage : 1.000         -> score 1.00
  COMPOSITE     : 0.823

Windhoek C
  Record length : 14.4 years  -> score 0.48
  Completeness  : 0.983         -> score 0.98
  PRCP coverage : 1.000         -> score 1.00
  COMPOSITE     : 0.811

Gobabis
  Record length : 55.0 years  -> score 1.00
  Completeness  : 0.609         -> score 0.61
  PRCP coverage : 0.602         -> score 0.60
  COMPOSITE     : 0.744

Keetmanshoop
  Record length : 15.4 years  -> score 0.51
  Completeness  : 0.930         -> score 0.93
  PRCP coverage : 1.000        

In [ ]:
# Step 1.4: Range Validation and Outlier Removal

NAMIBIA_BOUNDS = {
    'TMAX': (-5, 50),     # degrees Celsius
    'TMIN': (-10, 40),
    'PRCP': (0, 300),     # mm per day
    'AWND': (0, 50),      # m/s wind speed
}

def flag_outliers(df, station_name, bounds=NAMIBIA_BOUNDS):
    df = df.copy()
    flags = {}
    total_outliers = 0

    for col, (lo, hi) in bounds.items():
        if col in df.columns:
            mask = (df[col] < lo) | (df[col] > hi)
            count = mask.sum()
            flags[col] = count
            total_outliers += count
            df.loc[mask, col] = np.nan  # replace with NaN

    print(f"{station_name}")
    for col, count in flags.items():
        print(f"  {col}: {count} outliers removed")
    print(f"  Total outliers: {total_outliers}")
    print()
    return df


# --- Run on all stations and save to data/clean/
STATION_FILES = {
    'windhoek_main' : '../data/raw/windhoek_main_daily.csv',
    'windhoek_a'    : '../data/raw/windhoek_a_daily.csv',
    'windhoek_b'    : '../data/raw/windhoek_b_daily.csv',
    'windhoek_c'    : '../data/raw/windhoek_c_daily.csv',
    'gobabis'       : '../data/raw/gobabis_daily.csv',
    'keetmanshoop'  : '../data/raw/keetmanshoop_daily.csv',
    'mariental'     : '../data/raw/mariental_daily.csv',
    'rundu'         : '../data/raw/rundu_daily.csv',
    'ondangwa'      : '../data/raw/ondangwa_daily.csv',
    'walvis_bay'    : '../data/raw/walvis_bay_daily.csv',
}

STATION_NAMES = {
    'windhoek_main' : 'Windhoek Main (GSN)',
    'windhoek_a'    : 'Windhoek A',
    'windhoek_b'    : 'Windhoek B',
    'windhoek_c'    : 'Windhoek C',
    'gobabis'       : 'Gobabis',
    'keetmanshoop'  : 'Keetmanshoop',
    'mariental'     : 'Mariental',
    'rundu'         : 'Rundu',
    'ondangwa'      : 'Ondangwa',
    'walvis_bay'    : 'Walvis Bay',
}

clean_dfs = {}

for key, filepath in STATION_FILES.items():
    df = pd.read_csv(filepath, parse_dates=['DATE'])
    df_clean = flag_outliers(df, STATION_NAMES[key])
    df_clean.to_csv(f'../data/clean/{key}_daily_clean.csv', index=False)
    clean_dfs[key] = df_clean

print("=" * 50)
print("RANGE VALIDATION COMPLETE")
print("All clean files saved to data/clean/")
print("=" * 50)

Windhoek Main (GSN)
  TMAX: 0 outliers removed
  TMIN: 0 outliers removed
  PRCP: 0 outliers removed
  Total outliers: 0

Windhoek A
  PRCP: 0 outliers removed
  Total outliers: 0

Windhoek B
  PRCP: 0 outliers removed
  Total outliers: 0

Windhoek C
  PRCP: 0 outliers removed
  Total outliers: 0

Gobabis
  TMAX: 0 outliers removed
  TMIN: 0 outliers removed
  PRCP: 0 outliers removed
  Total outliers: 0

Keetmanshoop
  PRCP: 0 outliers removed
  Total outliers: 0

Mariental
  PRCP: 0 outliers removed
  Total outliers: 0

Rundu
  TMAX: 0 outliers removed
  TMIN: 0 outliers removed
  PRCP: 0 outliers removed
  Total outliers: 0

Ondangwa
  TMAX: 0 outliers removed
  TMIN: 0 outliers removed
  PRCP: 0 outliers removed
  Total outliers: 0

Walvis Bay
  TMAX: 0 outliers removed
  TMIN: 0 outliers removed
  PRCP: 0 outliers removed
  Total outliers: 0

RANGE VALIDATION COMPLETE
All clean files saved to data/clean/


In [ ]:
memo = """# Namibian Station Audit — NOAA ISD Data Quality Assessment

**Prepared by:** Wilka Igulu  
**Date:** June 2026  
**Data source:** NOAA NCEI Global Historical Climatology Network Daily (GHCND)  
**API endpoint:** https://www.ncei.noaa.gov/access/services/data/v1

---

## Background

As a first step in building the climate risk pipeline for the Goreangab Water 
Reclamation Plant, I pulled daily weather records for ten Namibian stations from 
the NOAA NCEI database. The goal was to understand what data is actually available 
before committing to any modelling choices downstream.

The station IDs listed in most references to NOAA's Integrated Surface Database 
(ISD) are not directly compatible with the GHCND daily-summaries API. Correct 
GHCND identifiers were obtained from the NOAA master stations file at:
https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt

---

## Station inventory

| Station | GHCND ID | Period | Years | Completeness | PRCP Coverage | DQS |
|---|---|---|---|---|---|---|
| Windhoek Main (GSN) | WA007401540 | 1970–2024 | 55.0 | 98.5% | 70.6% | **0.921** |
| Rundu | WA012084750 | 1970–2024 | 55.0 | 85.2% | 51.4% | **0.819** |
| Mariental | WA005688170 | 1970–1986 | 16.3 | 96.7% | 99.9% | **0.827** |
| Windhoek B | WA007401240 | 1970–1985 | 15.4 | 98.4% | 100.0% | **0.823** |
| Windhoek C | WA007401850 | 1970–1984 | 14.4 | 98.3% | 100.0% | **0.811** |
| Windhoek A | WA007400630 | 1970–1984 | 14.4 | 96.6% | 100.0% | **0.804** |
| Keetmanshoop | WA004192150 | 1970–1985 | 15.4 | 93.0% | 100.0% | **0.802** |
| Gobabis | WA007878380 | 1970–2024 | 55.0 | 60.9% | 60.2% | **0.744** |
| Walvis Bay | WAM00068098 | 1990–2024 | 34.9 | 58.3% | 6.6% | **0.600**  |
| Ondangwa | WAM00068006 | 1973–2024 | 51.9 | 35.0% | 15.9% | **0.530**  |
| Lüderitz | — | — | — | — | — | **NOT IN GHCND** |

The Data Quality Score (DQS) is a weighted composite: record length (0.35), 
completeness (0.40), and PRCP coverage (0.25). Scores range from 0 to 1.

---

## Range validation

All retrieved records were checked against physically plausible bounds for 
Namibia: TMAX (−5°C to 50°C), TMIN (−10°C to 40°C), PRCP (0–300 mm/day), 
and wind speed (0–50 m/s). Zero outliers were detected across all stations. 
This is consistent with NOAA NCEI applying its own automated QC before 
serving data through the API, flagged values are withheld rather than 
passed through as bad numbers.

---

## Flags

**Ondangwa (DQS = 0.530):** Marginally above the 0.50 threshold. Record 
coverage is only 35% of expected days, and precipitation data is sparse 
throughout. The station had no records at all during the 1990s. Any output 
derived from Ondangwa data should be treated as indicative only.

**Walvis Bay (DQS = 0.600):** Records begin only in 1990 and precipitation 
coverage is just 6.6%, reflecting the hyperarid coastal climate where rain 
events are so rare that the station may not systematically record them. 
Walvis Bay temperature and wind data are more reliable than its precipitation 
figures suggest.

**Lüderitz:** Absent from the GHCND database entirely. This is a genuine 
data gap in NOAA's Namibian coverage. Lüderitz is noted here for 
transparency but cannot be included in the analysis.

---

## Conclusion

Three stations are suitable for Extreme Value Theory modelling: Windhoek Main 
(WA007401540), Rundu (WA012084750), and Gobabis (WA007878380). These are the 
only stations with records spanning 1970 to the present day, which is the 
minimum length needed for credible return level estimation.

Windhoek Main is the primary station for this pipeline. It carries the highest 
DQS in the network (0.921), covers 55 years without interruption and is the 
station closest to the Goreangab Water Reclamation Plant the target asset. 
All EVT outputs in subsequent phases are derived from this station, with the 
DQS attached to every result as a transparency mechanism.

Rundu and Gobabis are retained as secondary stations for spatial 
cross-validation. Their lower PRCP coverage scores are noted and will be 
referenced where their data is used.
"""
import os

#creating the folder 
os.makedirs('../outputs/report', exist_ok=True)

with open('../outputs/report/station_audit_memo.md', 'w') as f:
    f.write(memo)

print("Station audit memo saved to outputs/report/station_audit_memo.md")

Station audit memo saved to outputs/report/station_audit_memo.md
